# Loss Masking for Reasoning Models

This notebook demonstrates `thinkpack.apply_mask` — the tool for tokenizing training conversations and controlling which sections contribute to the loss.

**The think-collapse problem:** fine-tuning a thinking model on standard instruction/response data (with no special handling) causes the model to stop generating `<think>` blocks, degrading reasoning capability. `apply_mask` solves this by keeping the think block present in the training sequence while masking its tokens so no gradient flows through them.

Two models are used throughout:
- **Qwen3** (`prefixed=False`) — the chat template always emits `<think>\n\n</think>` even when no reasoning is provided.
- **OLMo-3-Think** (`prefixed=True`) — the template only appends `<think>` at inference time, so `apply_mask` injects an empty think block during training for context alignment.

**What this notebook covers:**
1. `MaskType` flags — PROMPT, THINK, RESPONSE
2. No masking (`masked=None`) — naive baseline
3. Think masking (`MaskType.THINK`) — most common training mode
4. Response-only training (`MaskType.PROMPT | MaskType.THINK`)
5. Conversations with reasoning content
6. The returned HuggingFace `Dataset`

In [ ]:
import sys

# add the src directory to the path so thinkpack can be imported without installation
sys.path.insert(0, "../../src")

from transformers import AutoTokenizer

from thinkpack import MaskType, apply_mask, detect_model

In [ ]:
# load tokenizers for both models (cpu-only, no gpu needed)
tok_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)
tok_olmo = AutoTokenizer.from_pretrained(
    "allenai/OLMo-3-7B-Think",
    trust_remote_code=True,
)

# detect model properties — prefixed indicates whether the template injects <think>
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    info = detect_model(tok)
    print(f"{name:7s} — prefixed: {info.prefixed}, tag: {info.open_tag}")

## 1. `MaskType` Flags

`MaskType` is an `IntFlag` with three sections:

| Flag | Covers |
|---|---|
| `MaskType.PROMPT` | User instruction (everything before the think block) |
| `MaskType.THINK` | Full reasoning block, including opening and closing tags |
| `MaskType.RESPONSE` | Model answer (everything after the think block) |

Flags can be combined with `|`. Tokens where `label == -100` are ignored by PyTorch's `CrossEntropyLoss` and all major training frameworks (Trainer, SFTTrainer, unsloth).

Pass `masked=None` to train on all tokens with no masking applied.

In [ ]:
# helper: print each token and whether it is trained or masked
def show_labels(dataset, tokenizer, label: str) -> None:
    """Print token-level training status for the first record in a dataset."""
    input_ids = dataset[0]["input_ids"]
    labels = dataset[0]["labels"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    n_masked = labels.count(-100)
    n_trainable = len(labels) - n_masked
    print(f"=== {label} ===")
    print(f"  total={len(input_ids)}, masked={n_masked}, trainable={n_trainable}")
    for tok_str, lbl in zip(tokens, labels):
        status = "MASK " if lbl == -100 else "train"
        print(f"  [{status}] {repr(tok_str)}")
    print()

## 2. No Masking (`masked=None`) — Naive Baseline

All tokens contribute to the loss. For Qwen3, the template always emits an empty `<think>\n\n</think>` block — training on it teaches the model to produce empty thinking. For OLMo-3, the think block is absent entirely, so the training and inference contexts diverge. Both are examples of think collapse.

In [ ]:
# a minimal conversation with no reasoning key
conversation = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "4"},
]

# masked=None — no sections masked, all tokens trainable
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[conversation],
        tokenizer=tok,
        masked=None,
    )
    show_labels(ds, tok, f"{name} — masked=None")

## 3. Think Masking (`MaskType.THINK`) — Most Common Mode

The think block is included in the training sequence so the training and inference contexts match, but its tokens are masked so no gradient flows through them. The model trains on the instruction and the response only.

For conversations without a `"reasoning"` key, `apply_mask` automatically injects an empty think block — this is important for PREFIXED models like OLMo-3 that always emit a think block at inference time.

In [ ]:
# MaskType.THINK — think block masked, instruction and response trainable
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[conversation],
        tokenizer=tok,
        masked=MaskType.THINK,
    )
    show_labels(ds, tok, f"{name} — MaskType.THINK")

## 4. Response-Only Training (`MaskType.PROMPT | MaskType.THINK`)

A stricter variant: masks everything before the response. Only the response tokens contribute to the loss. The think block is still injected and present as context — the response is always predicted from `[instruction + think block]`, matching inference.

In [ ]:
# MaskType.PROMPT | MaskType.THINK — only response tokens are trainable
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[conversation],
        tokenizer=tok,
        masked=MaskType.PROMPT | MaskType.THINK,
    )
    show_labels(ds, tok, f"{name} — MaskType.PROMPT | MaskType.THINK")

## 5. Conversations with Reasoning Content

When the assistant message includes a `"reasoning"` key with actual content (e.g. distilled from a teacher model), `MaskType.THINK` masks the full reasoning block including its tags while the response remains trainable.

This is the intended pattern for distillation data — the model sees the reasoning as context and trains only on the response it derives from it.

In [ ]:
# conversation with actual reasoning content
reasoning_conversation = [
    {"role": "user", "content": "What is 2+2?"},
    {
        "role": "assistant",
        "content": "4",
        "reasoning": "I need to add 2 and 2. 2 + 2 = 4.",
    },
]

# the reasoning tokens are masked — only the response contributes to the loss
for name, tok in [("Qwen3", tok_qwen), ("OLMo-3", tok_olmo)]:
    ds = apply_mask(
        conversations=[reasoning_conversation],
        tokenizer=tok,
        masked=MaskType.THINK,
    )
    show_labels(ds, tok, f"{name} — real reasoning, MaskType.THINK")

## 6. The Returned `Dataset`

`apply_mask` returns a HuggingFace `Dataset` with three columns, ready to pass directly to a Trainer or SFTTrainer.

| Column | Description |
|---|---|
| `input_ids` | Token IDs for the full training sequence |
| `labels` | Same as `input_ids`, with masked positions set to `-100` |
| `attention_mask` | All-ones mask (no padding — sequences are variable-length) |

In [ ]:
# apply_mask over a batch of conversations — returns a HuggingFace Dataset
batch = [
    [
        {"role": "user", "content": "What is 2+2?"},
        {"role": "assistant", "content": "4", "reasoning": "2 + 2 = 4."},
    ],
    [
        {"role": "user", "content": "What is the capital of France?"},
        {
            "role": "assistant",
            "content": "Paris.",
            "reasoning": "France's capital is Paris.",
        },
    ],
]

ds = apply_mask(
    conversations=batch,
    tokenizer=tok_qwen,
    masked=MaskType.THINK,
)

print(f"dataset columns : {ds.column_names}")
print(f"num records     : {len(ds)}")
print(f"record 0 length : {len(ds[0]['input_ids'])} tokens")
print(f"record 1 length : {len(ds[1]['input_ids'])} tokens")

# the dataset can be passed directly to a Trainer or SFTTrainer
# trainer = SFTTrainer(model=model, train_dataset=ds, ...)

## Summary

| `masked` | Think block present? | Instruction trained? | Think trained? | Response trained? |
|---|---|---|---|---|
| `None` | Model-dependent | Yes | Yes (if present) | Yes |
| `MaskType.THINK` | Yes (injected if absent) | Yes | **No** | Yes |
| `PROMPT \| THINK` | Yes (injected if absent) | **No** | **No** | Yes |
| `MaskType.RESPONSE` | Model-dependent | Yes | Yes (if present) | **No** |

**Key points:**
- `apply_mask` accepts the same conversation format as `apply_chat_template` — a list of `{"role", "content"}` dicts, with an optional `"reasoning"` key on assistant messages.
- When masking is active and no `"reasoning"` key is present, an empty think block is injected automatically, keeping training context consistent with inference.
- Token boundaries are found by tokenizing text prefixes, which correctly handles models like Qwen3 that always emit a think block regardless of the reasoning content.
- `MaskType` flags can be combined freely with `|`.